In [2]:
import h5py,os,re,click
import numpy as np

ens='cE211.044.112'

folder='05_moments_run5_1DV'

path='data_aux/cfgs_sup_N'
with open(path,'r') as f:
    cfgs=f.read().splitlines()
    
basepath='data_aux/'
def load(inpath):
    with open(f'{basepath}{inpath}','r') as f:
        cfgs=f.read().splitlines()
    return cfgs

inpath=f'/p/project1/ngff/li47/code/scratch/run/{folder}/{ens}/data_avgmore/'
outpath=f'/p/project1/ngff/li47/code/scratch/run/{folder}/{ens}/data_merge_sup/'
os.makedirs(outpath,exist_ok=True)

def run():
    files=os.listdir(inpath+cfgs[0])
    for ifile,file in enumerate(files):
        if file.endswith('0,0,1,0,0,0.h5'):
            continue
        outfile=f'{outpath}/{file}'
        outfile_flag=outfile+'_flag'
        if os.path.isfile(outfile) and (not os.path.isfile(outfile_flag)):
            continue
        with open(outfile_flag,'w') as f:
            pass
        
        setupQ=True
        key2data={}
        for icfg,cfg in enumerate(cfgs):
            print(f'{ifile}/{len(files)}',f'{icfg}/{len(cfgs)}',end='               \r')
            infile=f'{inpath}/{cfg}/{file}'
            with h5py.File(infile) as f:
                if setupQ:
                    key2info={key:f[key][:] for key in f.keys() if key not in ['data']}
                    
                    key2info['notes']=[ele.decode() for ele in key2info['notes']]+['adding additional dimensions [cfg_sup_*] in the beginning']
                    for key in ['cfgs_sup_Njl','cfgs_sup_Njs','cfgs_sup_Njc','cfgs_sup_Njg']:
                        key2info[key]=load(key)
                        
                    setupQ=False
                    
                for key in f['data'].keys():
                    if key not in key2data.keys():
                        key2data[key]=[]
                    t=key[1]; t='l' if t=='+' else t
                    cfgs_sup=key2info[f'cfgs_sup_Nj{t}']
                    # print(key,t,len(cfgs_sup))
                    if cfg in cfgs_sup:
                        key2data[key].append(f[f'data/{key}'][:])
                    else:
                        print(cfg,key)
                
        with h5py.File(outfile,'w') as f:
            for key in key2info.keys():
                f.create_dataset(key,data=key2info[key])
            for key in key2data.keys():
                f.create_dataset(f'data/{key}',data=key2data[key])
                
        os.remove(outfile_flag)

run()